In [1]:
"""
This notebook is used to verify the configurator.py file.
"""

%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path
import yaml
import argparse
import numpy as np
import pprint

import matplotlib.pyplot as plt

import torch
from torch.utils.data import TensorDataset
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.strategies import DDPStrategy

from neurorient.model_pg_vol    import NeurOrientLightning
from neurorient.dataset         import TensorDatasetWithTransform, DictionaryDataset
from neurorient.logger          import Logger
from neurorient.image_transform import RandomPatch, PhotonFluctuation, PoissonNoise, GaussianNoise, BeamStopMask, BeamStopMask_from_file
from neurorient.configurator    import Configurator
# from neurorient.lr_scheduler    import CosineLRScheduler
from neurorient.config          import _CONFIG
from neurorient.utils_config    import (
    prepare_Slice2RotMat_config, prepare_IntensityNet_config, prepare_optimization_config)

torch.autograd.set_detect_anomaly(False)    # [WARNING] Making it True may throw errors when using bfloat16
                                            # Reference: https://discuss.pytorch.org/t/convolutionbackward0-returned-nan-values-in-its-0th-output/175571/4
                                            
logger = Logger()

/pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.26.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
from neurorient.reconstruction.slicing import get_real_mesh, gen_nonuniform_normalized_positions
from neurorient.utils_visualization import display_volumes, display_images

In [4]:
args = argparse.Namespace(
    yaml_file='yaml/PR772_downSampled_128x128.yaml', 
    file_name='PR772_neurorient_downsampled_128x128_filtered.pt', 
    checkpoint=None)

In [5]:

# %%
# [[[ HYPER-PARAMERTERS ]]]
# Load CONFIG from YAML
fl_yaml = args.yaml_file

with open(fl_yaml, 'r') as fh:
    config_dict = yaml.safe_load(fh)
CONFIG = Configurator.from_dict(config_dict)
logger.log(f"loaded configuration from yaml_file: {fl_yaml}.")

merged_config = CONFIG.merge_with_priority(_CONFIG, self_has_priority=True)
logger.log(f"overwrite default model configurations with customed configurations.")


if hasattr(merged_config.TRAINING, 'SEED'):
    L.seed_everything(merged_config.TRAINING.SEED)
    logger.log(f"SEED set to {merged_config.TRAINING.SEED}.")
else:
    logger.log(f"SEED not specified and not set.")

# ...Checkpoint
dir_chkpt           = Path(os.path.join(merged_config.TRAINING.BASE_DIRECTORY, merged_config.TRAINING.CHKPT_DIRECTORY))
dir_chkpt.mkdir(parents=True, exist_ok=True)
logger.log(f"checkpoints will be saved to {dir_chkpt}.")

# ...Dataset
dir_dataset       = merged_config.DATASET.DATASET_DIRECTORY
# necessary info to fetch data file name
pdb               = merged_config.DATASET.PDB
num_images        = merged_config.DATASET.NUM_IMG
if args.file_name is not None:
    data_file_name = args.file_name
else:
    data_file_name = f'{pdb}_increase1_poissonFalse_num{num_images//1000}K.pt'
logger.log(f'data read from {data_file_name}')

# necessary info to define datasets
if hasattr(merged_config.DATASET, 'FRAC_TOTAL'):
    frac_total        = merged_config.DATASET.FRAC_TOTAL
else:
    frac_total        = 1.0
frac_train        = merged_config.DATASET.FRAC_TRAIN
size_batch        = merged_config.DATASET.BATCH_SIZE
num_workers       = merged_config.DATASET.NUM_WORKERS

# ...Training
max_epochs           = merged_config.TRAINING.MAX_EPOCHS
num_gpus             = min(torch.cuda.device_count(), merged_config.TRAINING.NUM_GPUS)
logger.log(f'training the model with {max_epochs} epochs and {num_gpus} GPUs')

Seed set to 42


[2024-12-27 23:08:17] loaded configuration from yaml_file: yaml/PR772_downSampled_128x128.yaml.

[2024-12-27 23:08:17] overwrite default model configurations with customed configurations.

[2024-12-27 23:08:17] SEED set to 42.

[2024-12-27 23:08:17] checkpoints will be saved to /pscratch/sd/z/zhantao/neurorient_repo/experiments/PR772_downsampled_128x128.

[2024-12-27 23:08:17] data read from PR772_neurorient_downsampled_128x128_filtered.pt

[2024-12-27 23:08:17] training the model with 2000 epochs and 1 GPUs



In [6]:
spi_data = torch.load(os.path.join(dir_dataset, data_file_name))

# Set global seed and split data...
total_num_data    = len(spi_data['intensities'])
data              = spi_data['intensities'][:int(total_num_data * frac_total)]
spi_data_train    = data[:int(len(data) * frac_train) ]
spi_data_validate = data[ int(len(data) * frac_train):]

transform_list = []

if merged_config.DATASET.USES_PHOTON_FLUCTUATION:
    # set up photon fluctuation transformation
    photon_fluctuation = PhotonFluctuation(
        'neurorient/data/image_distribution_by_photon_count.npy',
        return_mask=False)
    transform_list.append(photon_fluctuation)
    logger.log(f'transformation: photon fluctuation applied to training and validation datasets.')


if merged_config.DATASET.USES_POISSON_NOISE:
    poisson_noise = PoissonNoise(return_mask=False)
    transform_list.append(poisson_noise)
    logger.log(f'transformation: poisson noise applied to training and validation datasets.')


if merged_config.DATASET.USES_GAUSSIAN_NOISE:
    gaussian_noise = GaussianNoise(sigma=merged_config.DATASET.GAUSSIAN_NOISE.SIGMA, return_mask=False)
    transform_list.append(gaussian_noise)
    logger.log(f'transformation: gaussian noise applied to training and validation datasets.')


if merged_config.DATASET.USES_BEAM_STOP_MASK:
    if hasattr(merged_config.DATASET.BEAM_STOP_MASK, 'READ_FILE'):
        beam_stop_file = merged_config.DATASET.BEAM_STOP_MASK.READ_FILE
        beam_stop_mask = BeamStopMask_from_file(file_path=beam_stop_file, return_mask=True)
        transform_list.append(beam_stop_mask)
        logger.log(f'transformation: beam stop mask loaded from {beam_stop_file}.')
    else:
        beam_stop_mask = BeamStopMask(width            = merged_config.DATASET.BEAM_STOP_MASK.WIDTH, 
                                    radius             = merged_config.DATASET.BEAM_STOP_MASK.RADIUS, 
                                    input_size         = data.shape[-2:],
                                    mask_orientation   = merged_config.DATASET.BEAM_STOP_MASK.ORIENTATION,
                                    return_mask        = True)
        transform_list.append(beam_stop_mask)
        logger.log(f'transformation: beam stop mask applied to training and validation datasets.')

# if merged_config.DATASET.USES_RANDOM_ROTATION:
#     import torchvision
#     random_rotation = RandomRotation(
#         degrees=(0, 360), return_mask=False,
#         interpolation=torchvision.transforms.InterpolationMode.BILINEAR
#     )
    
#     transform_list.append(random_rotation)
#     logger.log(f'transformation: using random rotation.')
    
if merged_config.DATASET.USES_RANDOM_PATCH:
    # set up random patch transformation
    num_patch       = merged_config.DATASET.PATCH.NUM_PATCHES
    size_patch_min  = merged_config.DATASET.PATCH.SIZE_PATCH_MIN
    size_patch_max  = merged_config.DATASET.PATCH.SIZE_PATCH_MAX
    random_patch = RandomPatch(num_patch       = num_patch,
                               size_patch_min  = size_patch_min,
                               size_patch_max  = size_patch_max,
                               return_mask     = True)
    transform_list.append(random_patch)
    logger.log(f'transformation: random patch applied to training and validation datasets.')
    
    
if len(transform_list) > 0:
    transform_list   = tuple(transform_list)
    _dataset_train    = TensorDatasetWithTransform(
        spi_data_train.unsqueeze(1), transform_list = transform_list, seed=merged_config.TRAINING.SEED)
    _dataset_validate = TensorDatasetWithTransform(
        spi_data_validate.unsqueeze(1), transform_list = transform_list, seed=merged_config.TRAINING.SEED)
    
    logger.log(f'{len(transform_list)} transformations applied to training and validation datasets.')
    
    train_data = {key: [] for key in _dataset_train[0].keys()}
    for i, d in enumerate(_dataset_train):
        for _key in d.keys():
            train_data[_key].append(d[_key])
    for _key in train_data.keys():
        train_data[_key] = torch.stack(train_data[_key], dim=0)
    dataset_train = DictionaryDataset(**train_data)
    del train_data
    
    validate_data = {key: [] for key in _dataset_validate[0].keys()}
    for i, d in enumerate(_dataset_validate):
        for _key in d.keys():
            validate_data[_key].append(d[_key])
    for _key in validate_data.keys():
        validate_data[_key] = torch.stack(validate_data[_key], dim=0)
    dataset_validate = DictionaryDataset(**validate_data)
    del validate_data
    
    logger.log(f'created dictionary datasets for training and validation.')
else:
    dataset_train    = TensorDataset(spi_data_train.unsqueeze(1))
    dataset_validate = TensorDataset(spi_data_validate.unsqueeze(1))
    logger.log(f'NO random patch transformation applied to training and validation datasets.')

logger.log(f'created training dataset with {len(dataset_train)} images and validation dataset with {len(dataset_validate)} images.')

[2024-12-27 23:08:18] transformation: beam stop mask loaded from /global/homes/z/zhantao/Projects/NeuralOrientationMatching/input/PR772/beam_stop_mask_downsampled_128x128.pt.

[2024-12-27 23:08:18] 1 transformations applied to training and validation datasets.

[2024-12-27 23:08:26] created dictionary datasets for training and validation.

[2024-12-27 23:08:26] created training dataset with 6215 images and validation dataset with 328 images.



In [7]:
# lightning will handle the samplers for those dataloaders
sampler_train    = None
dataloader_train = torch.utils.data.DataLoader( dataset_train,
                                                sampler     = sampler_train,
                                                shuffle     = True,
                                                pin_memory  = False,
                                                batch_size  = 5, drop_last=True)

sampler_validate    = None
dataloader_validate = torch.utils.data.DataLoader( dataset_validate,
                                                   sampler     = sampler_validate,
                                                   shuffle     = False,
                                                   pin_memory  = False,
                                                   batch_size  = 5, drop_last=True)


In [8]:
over_sampling = merged_config.MODEL.OVERSAMPLING
photons_per_pulse = merged_config.DATASET.INCREASE_FACTOR * 1e12
config_optimization = prepare_optimization_config(merged_config)
config_intensitynet = prepare_IntensityNet_config(merged_config)
config_slice2rotmat = prepare_Slice2RotMat_config(merged_config)

# config_slice2rotmat['point_group'] = merged_config.MODEL.POINT_GROUP

if hasattr(merged_config.MODEL, "PRED_PHOTON_PULSE_ANYWAY"):
    if merged_config.MODEL.PRED_PHOTON_PULSE_ANYWAY:
        use_fluctuation_predictor=True
    else:
        use_fluctuation_predictor=False
else:
    if merged_config.DATASET.USES_PHOTON_FLUCTUATION:
        use_fluctuation_predictor=True
    else:
        use_fluctuation_predictor=False
        
logger.log(f"Using fluctuation predictor: {use_fluctuation_predictor}")

if hasattr(merged_config.MODEL, "ROTMAT_DIVERSITY"):
    config_orientation_diversity_loss = {
        'max': merged_config.MODEL.ROTMAT_DIVERSITY.MAX,
        'min': merged_config.MODEL.ROTMAT_DIVERSITY.MIN,
        'scale': merged_config.MODEL.ROTMAT_DIVERSITY.SCALE
    }
else:
    config_orientation_diversity_loss = None
logger.log(f"config_orientation_diversity_loss: \n", config_orientation_diversity_loss)

[2024-12-27 23:08:26] Using fluctuation predictor: True

[2024-12-27 23:08:26] config_orientation_diversity_loss: 
 None



In [9]:

model = NeurOrientLightning(
    spi_data['pixel_position_reciprocal'],
    over_sampling=over_sampling, 
    photons_per_pulse=photons_per_pulse,
    use_bifpn=merged_config.MODEL.USE_BIFPN,
    use_fluctuation_predictor=use_fluctuation_predictor,
    config_slice2rotmat=config_slice2rotmat,
    config_intensitynet=config_intensitynet,
    config_optimization=config_optimization
)

logger.log( 
    'arguments being used in building the model:\n',
    f'over_sampling={over_sampling}\n',
    f'photons_per_pulse={photons_per_pulse:.2e}\n',
    'config_slice2rotmat: ', '\n', pprint.pformat(config_slice2rotmat), '\n',
    'config_optimization: ', '\n', pprint.pformat(config_optimization))

if args.checkpoint is not None:
    model.load_state_dict(
        torch.load(args.checkpoint)['state_dict']
    )
    logger.log(f"Resume training from state_dict of: {args.checkpoint}.")

# logger.log(
#     "model created with the following architecture:\n",
#     pprint.pformat(model)
# )

Initialized with 46080 reference rotations and max_val: 100.0
[2024-12-27 23:08:27] arguments being used in building the model:
 over_sampling=1.0
 photons_per_pulse=1.00e+12
 config_slice2rotmat:  
 {'pretrained': True, 'size': 18} 
 config_optimization:  
 {'loss_func': 'MSELoss',
 'lr': 0.0003,
 'scheduler': {'min_lr': 1e-07,
               'name': 'CosineLRScheduler',
               'total_epochs': 1000,
               'warmup_epochs': 5},
 'weight_decay': 1e-07}



In [10]:
# from neurorient.model_pg_vol import Slice2RotMat_CodeBook

# i2o_model = Slice2RotMat_CodeBook()
# model.to('cuda');

# i2o_model.to('cuda')

# i2o_model.get_ref_images(model.model.volume_predictor, model.model.pixel_position_reciprocal, model.model.image_dimension, model.model.over_sampling)

# d = i2o_model(torch.randn(30,128,128).to('cuda'))

# d.argmin(dim=1).shape

# min_encoding_indices = torch.argmin(d, dim=1).unsqueeze(1)
# min_encodings = torch.zeros(
#     min_encoding_indices.shape[0], i2o_model.n_ref).to(d.device)
# min_encodings.scatter_(1, min_encoding_indices, 1)

# min_encodings.shape

In [11]:
# model.load_state_dict(torch.load('/pscratch/sd/z/zhantao/neurorient_repo/experiments/PR772_downsampled_128x128/lightning_logs/version_26012435/checkpoints/last.ckpt')['state_dict'])

In [12]:
# batch = next(iter(dataloader_train))
# task_type = 'train'

# if isinstance(batch, dict):
#     slices_true = batch['image'].to(model.dtype).to(model.device)
#     input_mask  = batch['input_mask'].bool().to(model.device)
#     general_mask = batch['general_mask'].bool().to(model.device)
# else:
#     slices_true = batch[0].to(model.dtype).to(model.device)
#     input_mask = torch.ones_like(slices_true).bool().bool().to(model.device)
#     general_mask = torch.ones_like(slices_true).bool().bool().to(model.device)

# # Apply input and general masks and loss scale factor to get input slices.
# slices_input  = torch.log(input_mask * slices_true * model.model.loss_scale_factor + 1e-8)

# # predict orientations from images
# orientations_out = model.model.image_to_orientation(slices_input)
# if isinstance(orientations_out, tuple):
#     loss_vq, orientations, perplexity = orientations_out
# else:
#     orientations = orientations_out
#     loss_vq = torch.tensor(0.0).to(model.device)
#     perplexity = torch.tensor(0.0).to(model.device)
# # get reciprocal positions based on orientations
# # HKL has shape (3, num_qpts)
# HKL = gen_nonuniform_normalized_positions(
#     orientations, model.model.pixel_position_reciprocal, model.model.over_sampling)
# # predict slices from HKL
# _slices_pred = model.model.predict_slice(HKL).view((-1, 1,) + (model.model.image_dimension,)*2)

In [13]:
checkpoint_callback = ModelCheckpoint(
    every_n_train_steps=10, save_last=True, save_top_k=1, monitor="train_loss",
    filename=f'{pdb}-{{epoch}}-{{step}}'
)

torch.set_float32_matmul_precision('high')

trainer = L.Trainer(
    max_epochs=max_epochs, accelerator='gpu', 
    callbacks=[checkpoint_callback, TQDMProgressBar(refresh_rate=1)],
    log_every_n_steps=1, devices=num_gpus, sync_batchnorm = True,
    enable_checkpointing=True, default_root_dir=dir_chkpt)

# dump configuration to file for later reference
dump_yaml_fname = Path(os.path.join(trainer.logger.log_dir, 'input.yaml'))
dump_yaml_fname.parent.mkdir(parents=True, exist_ok=True)
merged_config.dump_to_file(dump_yaml_fname)

dump_log_fname = Path(os.path.join(trainer.logger.log_dir, 'log.txt'))
dump_log_fname.parent.mkdir(parents=True, exist_ok=True)
logger.dump_to_file(dump_log_fname)

trainer.fit(model, dataloader_train)

/pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site- ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
/pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site- ...
/pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site-packages/lightning/pytorch/trainer/configuration_validator.py:74: You defined a `validation_step` 

Epoch 0:   0%|          | 0/1243 [00:00<?, ?it/s] 

100%|██████████| 922/922 [01:51<00:00,  8.28it/s]


Reference images generated, sampling at max_val: 10.0
Epoch 0:   1%|          | 10/1243 [01:55<3:57:10,  0.09it/s, v_num=2.63e+7, train_loss=113.0]

/pscratch/sd/z/zhantao/conda/inr/lib/python3.9/site-packages/lightning/pytorch/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


ValueError: minvalue must be less than or equal to maxvalue

<Figure size 950x300 with 3 Axes>

In [13]:
# from neurorient.so3_decomposition import so3_point_group_operations
# from pytorch3d.transforms import so3_rotation_angle, random_rotations, so3_relative_angle


# class RotationFolding(torch.nn.Module):
#     def __init__(self, point_group):
#         super().__init__()
#         self.register_buffer('symm_ops', so3_point_group_operations(point_group))
        
#     def forward(self, rotations):
#         expanded_rotations = torch.einsum('gij, bjk -> bgik', self.symm_ops, rotations)
#         shape = expanded_rotations.shape[:2]
        
#         angles = so3_rotation_angle(expanded_rotations.reshape(-1,3,3)).view(shape)
        
#         min_encoding_indices = torch.argmin(angles, dim=1).unsqueeze(1)
#         min_encodings = torch.zeros(
#             min_encoding_indices.shape[0], len(self.symm_ops))
#         min_encodings.scatter_(1, min_encoding_indices, 1)

#         folded_rotations = torch.einsum('bg, bgik -> bik', min_encodings, expanded_rotations)
        
#         folded_rotations = rotations + (folded_rotations - rotations).detach()
        
#         return folded_rotations

# af_layer = RotationFolding('C3')